In [2]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
import glob
warnings.filterwarnings("ignore")

### Nielsen

In [3]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
xls = pd.ExcelFile(f'{dir}/../Data Source/Nielsen/Nielsen O+O Jun 26_270726.xlsx')
sheet_names = ['SG Nielsen Skincare', 'SG Nielsen Mass Medic']
dfs = {}
# Read each sheet into a DataFrame and store it in the dictionary
for sheet_name in sheet_names:
    dfs[sheet_name] = pd.read_excel(xls, sheet_name=sheet_name)

In [4]:
for df in dfs:
    print(df)

SG Nielsen Skincare
SG Nielsen Mass Medic


In [5]:
def month_to_number(month):
    months = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
    return months.get(month, month)

In [6]:
def map_platform(row):
    if row['Markets'] in ['Modern Trade/Singapore']:
        return 'Nielsen'
    elif row['Markets'] == 'Modern Trade/1 SMHM/Singapore':
        return 'NS Hypermarket & Supermarket'
    elif row['Markets'] == 'Modern Trade/2 PC & Western Pharmacy/Singapore':
        return 'NS Drugstore'
    elif row['Markets'] == 'Modern Trade/3 Convenience Store & Petrol Marts/Singapore':
        return 'NS CVS'
    else:
        return 'Others'
    


In [7]:
# Important!! Brand Mapping - Any new Brands need to be added here - LDB Brands - Make sure consistent for all O+O Scripts
## mass_medic = ['ACNE AID', 'ACNES', 'AVEENO', 'BENZAC', 'BIAFINE', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL', 'DERMATIX', 'DERMAVEEN', 'DR.G', 'DR.YU', 'EGO', 'LINOLA', 'MUSTELA', 'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'QV']
mass_medic = [
    'ACNE AID', 'ACNES', 'AVEENO', 'BALNEUM', 'BENZAC', 'BIO-OIL', 'CARMEX', 'CERAVE', 'CETAPHIL', 'CUREL',
    'DERMATIX', 'DERMAVEEN', 'DIFFERIN', 'DR.G', 'DR.YU', 'EGO', 'EUBOS', 'LACTACYD', 'LINOLA', 'MUSTELA',
    'NEUTROGENA', 'PANOXYL', 'PHYSIOGEL', 'SEBAMED', 'TOPICREM', 'VANICREAM', 'WIS', 'XHEKPON', 'FIRST AID BEAUTY',
    'AQUAPHOR', 'NOBACTER', 'LUBRIDERM', 'NEOSPORIN', 'DARROW', 'DEXERYL', 'ALERGIBON', 'ALPHYGIENE', 'BABIGOZ',
    'CANDERMYL', 'GALDERMA', 'GALDERMA OTHER', 'HELIOBLOC', 'HYDRODERM OMEGA', 'IOCON', 'IONIL', 'MACROLANE',
    'MICROBAN', 'MICROSUN', 'NESTLE', 'NUTRASPA', 'OBSERVANCE', 'PHYGIENE', 'R-GEN', 'SENTIAL', 'ACHE', 'ACNAID',
    'ACNE FREE', 'ACOFAR', 'ADDAX', 'AKILDIA', 'ALBOLENE', 'AMLACTIN', 'ANSEBIC', 'AQUA SOAP', 'AQUA-SOAP',
    'AVITIL', 'AZULENNE', 'BACCIDE', 'BEAUTY PLUS', 'BEDOOK', 'BEPANTHEN/BEPANTHOL', 'BETAGRANULOS', 'BIAFINE',
    'BIOBLAS', 'BIOCLIN', 'BIOLIQ', 'BIOXCIN', 'BLUE LIZARD', 'BODYSOL', 'BONAVEN', 'BOROLINE', 'CERAMOL',
    'CERTAIN DRI', 'CETOPIC', 'CHICCO', 'CICAMEL', 'COOPER', 'COTARYL', 'CRISTALIA', 'DECUBAL', 'DERMAC',
    'DERMACTIVE', 'DERMADRATE', 'DERMAGE', 'DERMAKERI', 'DERMENA', 'DERMON', 'DERSUPRIL', 'DEUMAVAN',
    'DOCTISSIMO PARAPHARMACIE', 'DR.LI', 'DR.LIDERMO', 'DRAYEX', 'DX2', 'E45', 'ELDOPAQUE', 'EMOLIENTA', 'EMOLIN',
    'EMOLIUM', 'EPIMAX', 'EVASOL', 'FARMOQUIMICA', 'FILTROSOL', 'FLUOCIN', 'FREI OEL (BOUHON)', 'GALENCO', 'GIFRER',
    'GILBERT', 'GOLD BOND', 'HAMILTON', 'HIDRAFIL', 'HIPOSOL', 'HYALIX', 'IDROVEL', 'IHADA', 'INFASIL',
    'INTERAPOTHEK', 'IRALTONE', 'ITANIDERM', 'KAMILODERM', 'KETOXIN', 'KINERASE', 'KORA', 'LACTIBON',
    'LACTO CALAMINE', 'LETI', 'LIFAR', 'LIPODERM', 'LOTRIMIN', 'MARQUE VERTE', 'MICRORET', 'MITOSYL', 'MODERM',
    'MULTIDERMOL', 'MUSSVITAL', 'NEUTRA LICE', 'NEUTRAPHARM', 'NORDIN', 'NUMIS', 'NUMIS MED', 'NURAPHARM',
    'NUTREM', 'NUTRISIL', 'OILATUM', 'OILLAN', 'OSMIN', 'OTC IBERICA', 'PANVEL DERMATIV', 'PARABOTICA',
    'PHARMACTIV', 'PHARMASEPT', 'PHISOHEX', 'PROCICAR', 'REGENERUM', 'RESTIV', 'RESTIVOIL', 'REVALESKIN', 'ROCHE',
    'ROGE CAVAILLES', 'ROYALCARE', 'RUGARD (SCHEFFLER)', 'SALILEX', 'SALLVE', 'SARNA', 'SAUGELLA', 'SEBORADIN',
    'SHADE', 'SMOOTH-E', 'SOLAR FOAM', 'S-OLE', 'SPECTRABAN', 'STANHOME FAMILY EXPERT', 'STIEFEL', 'STIEPROX',
    'STIPROX', 'STIPROXAL', 'TARMED', 'TRACTOPON', 'TRI DERMA MD', 'UREADERM', 'UVEIL-PS', 'UVESOL', 'VEA',
    'VENUSIA', 'VITA CITRAL', 'VITALIFE', 'ZODIAC','QV', 'BOBAI',
    'COLLAGE','DERMAREST','EPIZONE E','GLAMY LAB','LU MILD','NOLAVER','OXECURE','RIUP','SEBCUR','SELENGENA','SEROPIPE','SHAAN','STAR VILLE','STRONGVILLE','SYNOBAR','UREMOL',
    'URISEC','ZINPLEX'
]
print('Total mass_medic Brands: ', len(mass_medic))

Total mass_medic Brands:  217


In [8]:
def map_brand(row):
    if row['LOCAL BRAND'] in ['GARNIER', 'MAYBELLINE','3CE'] :
        return row['LOCAL BRAND']
    elif row['LOCAL BRAND'] == 'LOREAL DERMO EXPERTISE':
        return 'LOREAL PARIS'
    elif row['LOCAL BRAND'] in  mass_medic:
        return 'Mass Medic'
    elif pd.isna(row['LOCAL BRAND']):
        return 'Market'
    else:
        return 'Others'

In [9]:
def map_category(row):
    if row['GLOBAL SEGMENT'] == 'MALE' :
        return 'Male Skincare'
    elif row['GLOBAL SEGMENT'] == 'FEMALE' :
        return 'Female Skincare'
    else:
        return 'Others'

In [10]:
def map_value(row):
    if row['Platform'] == 'Nielsen' :
        return 0
    else:
        return row['Sales Value']

In [11]:
# Important!! Try to understand the filter and logic here
for df in dfs:
    dfs[df]['Year'] = dfs[df]['Periods'].str.extract('(\d+)', expand=False).astype(int) + 2000
    dfs[df]['Month Name'] = dfs[df]['Periods'].str.split().str[0]
    dfs[df]['Month'] = dfs[df]['Month Name'].apply(month_to_number)
    dfs[df]['Platform'] = dfs[df].apply(map_platform, axis=1)
    dfs[df]['Brand'] = dfs[df].apply(map_brand, axis=1)
    dfs[df]['Category'] = dfs[df].apply(map_category, axis=1)

In [12]:
# Aggregation for different nielsen offline reports
sg_nielsen_skincare = dfs['SG Nielsen Skincare'][~dfs['SG Nielsen Skincare']['Brand'].isin(['Others','Mass Medic'])].groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
sg_nielsen_skincare['Sales Value'] = sg_nielsen_skincare.apply(map_value, axis=1)
sg_nielsen_mass_medic = dfs['SG Nielsen Mass Medic'][dfs['SG Nielsen Mass Medic']['Brand'] == 'Mass Medic'].groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()

In [13]:
nielsen_sg_cpd = pd.concat([sg_nielsen_skincare, sg_nielsen_mass_medic], ignore_index=True)
nielsen_sg_cpd

,Platform,Year,Month,Brand,Category,Sales Value
0,NS CVS,2023,6,GARNIER,Female Skincare,481.020
1,NS CVS,2023,6,GARNIER,Male Skincare,122.100
2,NS CVS,2023,6,LOREAL PARIS,Male Skincare,37.300
3,NS CVS,2023,6,Market,Female Skincare,18820.449
4,NS CVS,2023,6,Market,Male Skincare,11081.607
...,...,...,...,...,...,...
970,Nielsen,2026,2,Mass Medic,Female Skincare,847606.220
971,Nielsen,2026,3,Mass Medic,Female Skincare,1096535.213
972,Nielsen,2026,4,Mass Medic,Female Skincare,834310.370
973,Nielsen,2026,5,Mass Medic,Female Skincare,913125.460


### OMT CPD

In [14]:
# Important!! Make sure the file exist and updated first
files = glob.glob(f'{dir}/../Data Source/OMT - O+O/SG CPD/*.xlsx')

# List to store DataFrames
dfs = []
# Read each file and append to the list
for file in files:
    df = pd.read_excel(file)
    dfs.append(df)

# Concatenate all DataFrames into one
online_data = pd.concat(dfs, ignore_index=True)
# mapping = pd.read_excel(f'{dir}/../Data Source/CPD Skincare Mapping/Skincare Mapping.xlsx', sheet_name='SG')

In [15]:
mapping = pd.read_excel(f'{dir}/../Data Source/CPD Skincare Mapping/Skincare Mapping.xlsx', sheet_name='SG')
print(mapping)
# Catgory mapping
def map_category(row):
    if (row['Category L1'] == 'MAKEUP') & (row['Category L2_x'] in ['EYE MAKEUP','FACE MAKEUP','LIP MAKEUP', 'NAIL MAKEUP', 'OTHER MAKEUP']):
        return 'Makeup'
    elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['FACE CARE & CLEANSING']):
        return 'Skincare'
    elif (row['Category L1'] == 'SKIN CARE') & (row['Category L2_x'] in ['SUN CARE']) & (row['Category L3'] in ['FACE PROTECTION']):
        return 'Suncare'
    elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] == 'HAIR COLOR'):
        return 'Hair Colour'
    elif (row['Category L1'] == 'HAIR') & (row['Category L2_x'] in ['HAIR CARE']):
        return 'Hair Care'
    else:
        return 'Others'

      Brand  Year  Month  Shopee  Lazada  Tiktok
0    Market  2021      1   0.955   0.955   0.955
1    Market  2021      2   0.955   0.955   0.955
2    Market  2021      3   0.955   0.955   0.955
3    Market  2021      4   0.955   0.955   0.955
4    Market  2021      5   0.955   0.955   0.955
..      ...   ...    ...     ...     ...     ...
331     3CE  2026      8   1.000   1.000   1.000
332     3CE  2026      9   1.000   1.000   1.000
333     3CE  2026     10   1.000   1.000   1.000
334     3CE  2026     11   1.000   1.000   1.000
335     3CE  2026     12   1.000   1.000   1.000

[336 rows x 6 columns]


In [16]:
def map_brand(row):
    if row['Brand_x'] == "L'OREAL PARIS" :
        return 'LOREAL PARIS'
    else:
        return row['Brand_x']

In [17]:
# Renaming columns for duplicated columns
online_data = online_data.rename(columns={'Mall Type': 'Platform', 'Total Est. Sales Local': 'Sales Value', 'Brand': 'Brand_x', 'Category L2': 'Category L2_x'})

In [18]:
# Important!! Try to understand the filter and logic here
online_data = online_data[online_data['Platform'].isin(['Shopee Mall', 'Lazada Mall'])]
online_data = online_data[online_data['Category L1'] != 'FRAGRANCE']
online_data['Brand_x'] = online_data.apply(map_brand, axis=1)
online_data[['Year', 'Month']] = online_data['Year Month'].str.split('-', expand=True)
online_data['Brand'] = 'Market'
online_data['Category'] = online_data.apply(map_category, axis=1)
online_data

,Country,Year Month,Universe,Platform,Category L1,Category L2_x,Category L3,Benefits,Formats,Brand_x,Product,Sales Value,Total units sold,Loreal 1P Est Sales Local,Year,Month,Brand,Category
0,SG,2024-01,MASS,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,FACE MASK & PACKS,HYDRATING,SHEET,TORRIDEN,[Torriden Official] DIVE IN Low Molecular Hyal...,40492.80,2109.0,NaN,2024,01,Market,Skincare
1,SG,2024-01,MASS,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,SERUM & ESSENCE,HYDRATING,SERUM,SKIN1004,[SKIN1004 1+1 EVENT] Madagascar Centella Ampou...,36658.44,1638.0,NaN,2024,01,Market,Skincare
2,SG,2024-01,MASS,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,SERUM & ESSENCE,BRIGHTENING,SERUM & ESSENCE,TORRIDEN,[Torriden Official] DIVE IN Low Molecular Hyal...,36341.76,2366.0,NaN,2024,01,Market,Skincare
3,SG,2024-01,MASS,Shopee Mall,SKIN CARE,SUN CARE,FACE PROTECTION,HYDRATING,SERUM,SKIN1004,SKIN1004 Madagascar Centella Hyalu-Cica Water-...,31720.62,1398.0,NaN,2024,01,Market,Suncare
4,SG,2024-01,MASS,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,SERUM & ESSENCE,SKIN BARRIER,SERUM & ESSENCE,COSRX,"[COSRX] The 6 Peptide Skin Booster Serum 30ml,...",30610.00,6122.0,NaN,2024,01,Market,Skincare
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
847840,SG,2026-06,MASS,Lazada Mall,MAKEUP,NAIL MAKEUP,OTHER NAIL MAKEUP,OTHER NAIL MAKEUP,OTHER NAIL MAKEUP,UR SUGAR,UR SUGAR 3D Floral Nail Art Stickers Spring De...,1.13,1.0,NaN,2026,06,Market,Makeup
847841,SG,2026-06,MASS,Lazada Mall,MAKEUP,NAIL MAKEUP,OTHER NAIL MAKEUP,OTHER NAIL MAKEUP,OTHER NAIL MAKEUP,BORN PRETTY,BORN PRETTY 1pc Nail Art Curve Tweezers Rhines...,0.92,1.0,NaN,2026,06,Market,Makeup
847842,SG,2026-06,MASS,Lazada Mall,SKIN CARE,FACE CARE & CLEANSING,FACE MASK & PACKS,OIL CONTROL,SHEET,BSKM,BSKM 25ml Fruit Mask Facial Hydrating Moisturi...,0.84,1.0,NaN,2026,06,Market,Skincare
847843,SG,2026-06,MASS,Shopee Mall,MAKEUP,OTHER MAKEUP,MAKEUP ACCESSORIES,MAKEUP ACCESSORIES,MAKEUP ACCESSORIES,MOLLIE LIPPER,Mollie Lipper 10pcs Eyelash Glue bottle bloc...,0.80,1.0,NaN,2026,06,Market,Makeup


In [19]:
# Filter and aggregate online data for market and brand
online_market = online_data.groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
loreal_brand = online_data[online_data['Brand_x'].isin(['GARNIER','MAYBELLINE','LOREAL PARIS','3CE'])].groupby(['Platform', 'Year', 'Month', 'Brand_x', 'Category'])['Sales Value'].sum().reset_index()
loreal_brand = loreal_brand.rename(columns={'Brand_x': 'Brand', 'Sales Value': 'Sales Value'})


In [20]:
# Merge skincare data for female and male percentage split
sg_skincare_split = pd.concat([online_market[online_market['Category'] == 'Skincare'], loreal_brand[loreal_brand['Category'] == 'Skincare']], ignore_index=True)
sg_skincare_split.head(3)

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Market,Skincare,429006.93
1,Lazada Mall,2024,02,Market,Skincare,509250.63
2,Lazada Mall,2024,03,Market,Skincare,699425.16


In [21]:
# SQL Query for male female percentage split using mapping file
query = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                'Female Skincare' AS Category
            FROM mapping
        
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                'Male Skincare' AS Category
            FROM mapping
        )

        SELECT
            a.Platform,
            a.Year,
            a.Month,
            a.Brand,
            b.Category,
            CASE
                WHEN a.Platform = 'Shopee Mall' THEN a."Sales Value" * b.Shopee
                WHEN a.Platform = 'Lazada Mall' THEN a."Sales Value" * b.Lazada
            END AS "Sales Value"
        FROM sg_skincare_split a
            LEFT JOIN mapping_tx b
                ON (
                    a.Brand = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

sg_skincare = sqldf(query)
sg_skincare.head()

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Market,Male Skincare,12251.711445
1,Lazada Mall,2024,01,Market,Female Skincare,416755.218555
2,Lazada Mall,2024,02,Market,Male Skincare,14543.335633
3,Lazada Mall,2024,02,Market,Female Skincare,494707.294367
4,Lazada Mall,2024,03,Market,Male Skincare,19974.398170


In [22]:
# Exclude skincare from market and brand
online_market = online_market[~(online_market['Category'].isin(['Skincare', 'Others']))]
loreal_brand = loreal_brand[~(loreal_brand['Category'].isin(['Skincare', 'Others']))]

In [23]:
# Merge Everything back
sg_online_cpd = pd.concat([online_market, loreal_brand, sg_skincare], ignore_index=True)
sg_online_cpd.tail(3)

,Platform,Year,Month,Brand,Category,Sales Value
1193,Shopee Mall,2026,06,LOREAL PARIS,Female Skincare,34565.897507
1194,Shopee Mall,2026,06,MAYBELLINE,Male Skincare,0.000000
1195,Shopee Mall,2026,06,MAYBELLINE,Female Skincare,1732.620000


### OMT LDB

In [24]:
# Important!! Make sure the file exist and updated first
dir = os.getcwd()
files = glob.glob(f'{dir}/../Data Source/OMT - O+O//SG LDB/*.xlsx')

# List to store DataFrames
dfs = []
# Read each file and append to the list
for file in files:
    df = pd.read_excel(file)
    dfs.append(df)

# Concatenate all DataFrames into one
ldb_data = pd.concat(dfs, ignore_index=True)

In [25]:
# Rename Platform
def map_platform(row):
    if row['Mall Type'] == 'Shopee Mall' :
        return 'Shopee Mall'
    elif row['Mall Type'] == 'Lazada Mall' :
        return 'Lazada Mall'
    elif row['Mall Type'] == 'Tiktok Mall' :
        return 'Tiktok Mall'
    else:
        return 'Others'

In [26]:
# Renaming columns for duplicated columns
ldb_data = ldb_data.rename(columns={'Total Est. Sales Local': 'Sales Value', 'Brand': 'Brand_x', 'Category L2': 'Category L2_x'})
print(ldb_data.columns[ldb_data.columns.duplicated()])
print(ldb_data.columns.tolist())



Index([], dtype='str')
['Country', 'Year Month', 'Universe', 'Mall Type', 'Category L1', 'Category L2_x', 'Category L3', 'Benefits', 'Formats', 'Brand_x', 'Product', 'Sales Value', 'Total units sold', 'Loreal 1P Est Sales Local']


In [27]:
# Important!! Try to understand the filter and logic here
ldb_data = ldb_data[ldb_data['Brand_x'].isin(mass_medic)]
ldb_data = ldb_data[ldb_data['Category L1'] == 'SKIN CARE']
ldb_data = ldb_data[ldb_data['Mall Type'].isin(['Shopee Mall', 'Lazada Mall'])]
ldb_data = ldb_data[
    ~(
        (ldb_data['Category L2_x'] == 'BODY CARE') |
        (
            (ldb_data['Category L2_x'] == 'SUN CARE') &
            (ldb_data['Category L3'] == 'FACE PROTECTION')
        )
    )
]
ldb_data['Platform'] = ldb_data.apply(map_platform, axis=1)
ldb_data[['Year', 'Month']] = ldb_data['Year Month'].str.split('-', expand=True)
ldb_data['Brand'] = 'Mass Medic'
ldb_data['Category'] = 'Female Skincare'
ldb_data.head()

,Country,Year Month,Universe,Mall Type,Category L1,Category L2_x,Category L3,Benefits,Formats,Brand_x,Product,Sales Value,Total units sold,Loreal 1P Est Sales Local,Platform,Year,Month,Brand,Category
17,SG,2024-01,MASS MEDICAL,Lazada Mall,SKIN CARE,FACE CARE & CLEANSING,FACIAL CLEANSER,HYDRATING,FACIAL CLEANSER,CETAPHIL,CETAPHIL Gentle Skin Cleanser 1000ML Bundle of...,5933.40,60.0,NaN,Lazada Mall,2024,01,Mass Medic,Female Skincare
20,SG,2024-01,MASS MEDICAL,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,FACIAL CLEANSER,HYDRATING,FACIAL CLEANSER,CETAPHIL,CETAPHIL Gentle Skin Cleanser 1000ML Bundle of...,5736.20,58.0,NaN,Shopee Mall,2024,01,Mass Medic,Female Skincare
21,SG,2024-01,MASS MEDICAL,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,FACIAL MOISTURIZER,HYDRATING,GEL,NEUTROGENA,[Bundle of 3] Neutrogena Hydrating Hydro Boost...,5652.72,108.0,NaN,Shopee Mall,2024,01,Mass Medic,Female Skincare
22,SG,2024-01,MASS MEDICAL,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,FACIAL CLEANSER,HYDRATING,FACIAL CLEANSER,CETAPHIL,CETAPHIL Gentle Skin Cleanser 1000ML Twin Pack...,5620.60,116.0,NaN,Shopee Mall,2024,01,Mass Medic,Female Skincare
31,SG,2024-01,MASS MEDICAL,Shopee Mall,SKIN CARE,FACE CARE & CLEANSING,FACIAL MOISTURIZER,SKIN BARRIER,CREAM,CETAPHIL,Cetaphil PRO AD Derma Skin Restoring Moisturiz...,4651.20,68.0,NaN,Shopee Mall,2024,01,Mass Medic,Female Skincare


In [28]:
# Aggregate LDB data
ldb_market = ldb_data.groupby(['Platform', 'Year', 'Month', 'Brand', 'Category'])['Sales Value'].sum().reset_index()
ldb_market.head()

,Platform,Year,Month,Brand,Category,Sales Value
0,Lazada Mall,2024,01,Mass Medic,Female Skincare,65733.12
1,Lazada Mall,2024,02,Mass Medic,Female Skincare,72747.58
2,Lazada Mall,2024,03,Mass Medic,Female Skincare,123107.20
3,Lazada Mall,2024,04,Mass Medic,Female Skincare,69523.17
4,Lazada Mall,2024,05,Mass Medic,Female Skincare,87275.03


### Final Transformation

In [29]:
# oo_sg_cpd = pd.concat([nielsen_sg_cpd], ignore_index=True)
oo_sg_cpd = pd.concat([nielsen_sg_cpd, online_market, loreal_brand, sg_skincare, ldb_market], ignore_index=True)
# oo_sg_cpd = pd.concat([nielsen_sg_cpd], ignore_index=True)
oo_sg_cpd = oo_sg_cpd.sort_values(by=['Platform', 'Year', 'Month', 'Brand', 'Category'])
oo_sg_cpd ['Month'] = oo_sg_cpd ['Month'].astype(int)
oo_sg_cpd ['Year'] = oo_sg_cpd ['Year'].astype(int)
oo_sg_cpd = oo_sg_cpd[oo_sg_cpd['Year'] >= 2021]

In [30]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()
year = last_month.year

# Format the output as "MMM YYYY"
filemonth = last_month.strftime("%b %Y").upper()

# Print the result
print(f"{filemonth}")

JUN 2026


In [31]:
if not os.path.exists(f'../Generated Data/O+O/{filemonth}'):
        os.makedirs(f'../Generated Data/O+O/{filemonth}')

In [32]:
oo_sg_cpd.to_excel(f'../Generated Data/O+O/{filemonth}/SG CPD {filemonth} O+O.xlsx', index=False)